# LeetCode 1414 — Find the Minimum Number of Fibonacci Numbers Whose Sum Is K bằng Python

Link đề: https://leetcode.com/problems/find-the-minimum-number-of-fibonacci-numbers-whose-sum-is-k/ · Độ khó: **Medium** · Chủ đề: Math, Greedy

Skeleton LeetCode:

```python
class Solution:
    def findMinFibonacciNumbers(self, k: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Nhận ra bài là **đổi tiền (coin change)** với "mệnh giá" là các số Fibonacci, và giải được bằng DP cho `k` nhỏ (baseline).
2. Hiểu và **chứng minh** tham lam "lấy số Fibonacci lớn nhất ≤ k" là tối ưu (định lý Zeckendorf và hai đẳng thức `2Fₙ = Fₙ₊₁ + Fₙ₋₂`, `Fₙ + Fₙ₊₁ = Fₙ₊₂`).
3. Cài đặt tham lam theo nhiều kiểu: vòng lặp trên danh sách, đệ quy, `bisect`, và đi lùi trên dãy Fibonacci với bộ nhớ `O(1)`.
4. Biết vì sao tham lam **không** đúng với hệ tiền bất kỳ (ví dụ `{1, 3, 4}`), và dùng DP để kiểm chứng.
5. Tránh lỗi: sinh thiếu số Fibonacci bằng đúng `k`, lệch chỉ số khi dùng `bisect`, sinh Fibonacci bằng đệ quy không nhớ.

## 2. Đề bài & ví dụ

Cho số nguyên `k`, trả về **số lượng ít nhất** các số Fibonacci có tổng bằng `k`. Mỗi số Fibonacci được dùng **nhiều lần**.

Dãy Fibonacci: `F₁ = 1`, `F₂ = 1`, `Fₙ = Fₙ₋₁ + Fₙ₋₂` (n > 2) ⇒ `1, 1, 2, 3, 5, 8, 13, 21, …`

Đảm bảo luôn tìm được (vì có số 1).

**Ví dụ 1:** `k = 7` → `2` (`2 + 5`).

**Ví dụ 2:** `k = 10` → `2` (`2 + 8`).

**Ví dụ 3:** `k = 19` → `3` (`1 + 5 + 13`).

**Ràng buộc:** `1 <= k <= 10^9`.

In [1]:
import random
import time
from bisect import bisect_right
from functools import lru_cache
from typing import List

examples = [(7, 2), (10, 2), (19, 3)]
for k, want in examples:
    print(f"k={k} -> {want}")

k=7 -> 2
k=10 -> 2
k=19 -> 3


## 3. Kiến thức nền

### 3.1 Sinh dãy Fibonacci đến giới hạn

Dùng hai biến `a, b` và gán đồng thời `a, b = b, a + b`. Tới `10^9` chỉ có khoảng **44** số — rất ít.

In [2]:
def fibs_upto(limit):
    res = []
    a, b = 1, 1
    while a <= limit:
        res.append(a)
        a, b = b, a + b
    return res

f = fibs_upto(10**9)
print(len(f), f[:12], "...", f[-1])

44 [1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144] ... 701408733


### 3.2 Bài toán đổi tiền (coin change) bằng DP

`dp[x]` = số đồng ít nhất để tạo tổng `x`; `dp[0] = 0`; `dp[x] = 1 + min(dp[x − c])` với mọi mệnh giá `c ≤ x`. Time `O(x_max · số_mệnh_giá)`.

In [3]:
def min_coins(coins, target):
    INF = float("inf")
    dp = [0] + [INF] * target
    for x in range(1, target + 1):
        for c in coins:
            if c <= x and dp[x - c] + 1 < dp[x]:
                dp[x] = dp[x - c] + 1
    return dp[target]

print(min_coins([1, 3, 4], 6), min_coins([1, 2, 3, 5, 8], 19))

2 3


### 3.3 `bisect_right` để tìm phần tử lớn nhất ≤ x

Trong danh sách tăng dần `arr`, `bisect_right(arr, x)` trả vị trí chèn **sau** mọi phần tử ≤ x ⇒ `arr[bisect_right(arr, x) − 1]` là phần tử lớn nhất ≤ x.

In [4]:
arr = [1, 2, 3, 5, 8, 13]
for x in [1, 7, 8, 12, 13, 100]:
    i = bisect_right(arr, x) - 1
    print(f"lớn nhất <= {x:3d}: {arr[i]}")

lớn nhất <=   1: 1
lớn nhất <=   7: 5
lớn nhất <=   8: 8
lớn nhất <=  12: 8
lớn nhất <=  13: 13
lớn nhất <= 100: 13


### 3.4 Hai đẳng thức Fibonacci quan trọng

- **Gộp hai số liền kề:** `Fₙ + Fₙ₊₁ = Fₙ₊₂` (định nghĩa).
- **Tách số lặp:** `2Fₙ = Fₙ₊₁ + Fₙ₋₂` (vì `Fₙ₊₁ = Fₙ + Fₙ₋₁` và `Fₙ = Fₙ₋₁ + Fₙ₋₂`).

Cả hai biến đổi **không làm tăng** số lượng số hạng (gộp còn làm giảm 1). Đây là chìa khoá chứng minh tham lam.

In [5]:
F = [0, 1, 1]
while len(F) < 20:
    F.append(F[-1] + F[-2])
print(all(F[n] + F[n + 1] == F[n + 2] for n in range(1, 17)))
print(all(2 * F[n] == F[n + 1] + F[n - 2] for n in range(3, 18)))
print("ví dụ: 2*8 =", 2 * 8, "= 13 + 3 =", 13 + 3)

True
True
ví dụ: 2*8 = 16 = 13 + 3 = 16


### 3.5 Định lý Zeckendorf (phát biểu)

Mọi số nguyên dương viết được **duy nhất** thành tổng các số Fibonacci **phân biệt, không có hai số liền kề** trong dãy (`F₂, F₃, …`). Ví dụ `19 = 13 + 5 + 1`, `100 = 89 + 8 + 3`. Biểu diễn này tìm được bằng tham lam và có **ít số hạng nhất**.

In [6]:
def zeckendorf(k):
    res = []
    for x in reversed(fibs_upto(k)):
        if x <= k:
            res.append(x)
            k -= x
    return res

for k in [19, 100, 7, 10]:
    print(k, "=", " + ".join(map(str, zeckendorf(k))))

19 = 13 + 5 + 1
100 = 89 + 8 + 3
7 = 5 + 2
10 = 8 + 2


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 3** `k = 19`. Fibonacci ≤ 19: `1, 1, 2, 3, 5, 8, 13`.

- Thử lấy số lớn nhất: `19 − 13 = 6`; lớn nhất ≤ 6 là `5`: `6 − 5 = 1`; lấy `1`. ⇒ `13 + 5 + 1` = **3** số.
- Có cách nào 2 số không? Cần `x + y = 19` với `x, y` Fibonacci: `13 + 6` (6 không phải), `8 + 11` (không), … ⇒ không. Vậy 3 là tối ưu.

**Ví dụ 1** `k = 7`: lớn nhất ≤ 7 là `5`, còn `2` ⇒ `5 + 2` = 2 số ✔. **Ví dụ 2** `k = 10`: `8 + 2` ✔.

Với hệ tiền thông thường, tham lam có thể sai (ví dụ `{1, 3, 4}`, tổng 6: tham lam `4 + 1 + 1` = 3, tối ưu `3 + 3` = 2). Fibonacci **đặc biệt** nhờ hai đẳng thức ở 3.4.

In [7]:
k = 19
steps = []
while k:
    x = max(f for f in fibs_upto(k))
    steps.append(x)
    print(f"còn {k:2d} -> lấy {x}")
    k -= x
print("dùng", len(steps), "số:", steps)

còn 19 -> lấy 13
còn  6 -> lấy 5
còn  1 -> lấy 1
dùng 3 số: [13, 5, 1]


## 5. Cách 1 — Brute Force / baseline: DP đổi tiền

### Ý tưởng
Coi các số Fibonacci là mệnh giá, chạy DP đổi tiền `dp[0..k]`. Chắc chắn đúng (xét mọi cách), nhưng tốn `O(k · log k)` thời gian và `O(k)` bộ nhớ.

In [8]:
def fib_dp(k):
    coins = sorted(set(fibs_upto(k)))       # bỏ số 1 trùng
    INF = k + 1
    dp = [0] + [INF] * k
    for x in range(1, k + 1):
        best = INF
        for c in coins:
            if c > x:
                break
            if dp[x - c] + 1 < best:
                best = dp[x - c] + 1
        dp[x] = best
    return dp[k]

for k, want in examples:
    print(k, "->", fib_dp(k), "| mong đợi", want)

7 -> 2 | mong đợi 2
10 -> 2 | mong đợi 2
19 -> 3 | mong đợi 3


### Độ phức tạp và nút thắt

- Time `O(k · L)` với `L ≈ 44` số Fibonacci ≤ k; Space `O(k)`.
- `k = 10^9` ⇒ ~4·10¹⁰ phép tính và mảng 10⁹ phần tử (~8 GB với list Python) — **không khả thi**.

Đo thử với `k` nhỏ:

In [9]:
for k in [10**3, 10**4, 10**5]:
    t = time.perf_counter()
    r = fib_dp(k)
    print(f"k={k:>6d}: {r} ({time.perf_counter() - t:.3f}s)")

k=  1000: 2 (0.001s)
k= 10000: 6 (0.016s)
k=100000: 9 (0.109s)


Nút thắt: DP tính đáp án cho **mọi** `x ≤ k`, trong khi ta chỉ cần `k`. Nếu có quy tắc chọn **chắc chắn đúng** ở mỗi bước, ta chỉ cần ~44 bước.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight — tham lam an toàn với Fibonacci.** Gọi `F` = số Fibonacci lớn nhất ≤ `k`. Khi đó **luôn có một lời giải tối ưu chứa `F`** ⇒ lấy `F`, giải tiếp `k − F`.

Lý do (chi tiết ở mục 10):
1. Từ một lời giải tối ưu bất kỳ, dùng `2Fₙ = Fₙ₊₁ + Fₙ₋₂` và `Fₙ + Fₙ₊₁ = Fₙ₊₂` để biến nó thành lời giải **không lặp, không liền kề** mà số lượng không tăng.
2. Tổng của các Fibonacci phân biệt, không liền kề, **đều nhỏ hơn `F`** thì **nhỏ hơn `F`** (thậm chí ≤ `F − 1`).
3. Vậy nếu lời giải đó không chứa `F` (và không thể chứa số lớn hơn `F` vì lớn hơn `k`), tổng < `F ≤ k` — mâu thuẫn.

**Quy tắc "nếu… thì…":** nếu `k > 0` thì lấy số Fibonacci lớn nhất ≤ `k`, đếm +1, `k −= F`.

**Bất biến:** số đếm hiện tại + đáp án tối ưu của `k` còn lại = đáp án tối ưu của `k` ban đầu.

## 7. Các cách tiếp theo

### Cách 2 — Tham lam lặp trên danh sách Fibonacci (duyệt từ lớn đến nhỏ)

**Từng đoạn:**
1. Sinh danh sách Fibonacci ≤ `k`.
2. Duyệt từ lớn đến nhỏ; nếu `f ≤ k` thì `k −= f`, đếm +1. (Mỗi số dùng tối đa một lần là đủ — tham lam không bao giờ cần lấy lại cùng số, vì `k − F < Fₙ₋₁` sau khi lấy `F = Fₙ`.)
3. Dừng khi `k == 0`.

In [10]:
def fib_greedy(k):
    fibs = fibs_upto(k)
    cnt = 0
    for f in reversed(fibs):
        if f <= k:
            k -= f
            cnt += 1
            if k == 0:
                break
    return cnt

for k, want in examples:
    print(k, "->", fib_greedy(k), "| mong đợi", want)

7 -> 2 | mong đợi 2
10 -> 2 | mong đợi 2
19 -> 3 | mong đợi 3


Time `O(log k)` (≈ 44 số, vì Fibonacci tăng theo cấp số nhân `φⁿ`), Space `O(log k)`.

### Cách 3 — Đệ quy (theo gợi ý của đề)

`f(k) = 0` nếu `k = 0`; ngược lại `f(k) = 1 + f(k − F)` với `F` lớn nhất ≤ `k`. Độ sâu đệ quy ≤ ~44.

In [11]:
FIBS = fibs_upto(10**9)

def fib_rec(k):
    if k == 0:
        return 0
    F = FIBS[bisect_right(FIBS, k) - 1]
    return 1 + fib_rec(k - F)

for k, want in examples:
    print(k, "->", fib_rec(k), "| mong đợi", want)

7 -> 2 | mong đợi 2
10 -> 2 | mong đợi 2
19 -> 3 | mong đợi 3


Ở đây dùng `bisect_right` trên danh sách sinh sẵn để tìm `F` trong `O(log 44)`. Time `O(log k · log log k)`, Space `O(log k)` (ngăn xếp đệ quy).

### Cách 4 — `bisect` lặp (không đệ quy)

Giống Cách 3 nhưng viết vòng lặp, dùng danh sách Fibonacci sinh **một lần** cho mọi truy vấn — tiện khi phải trả lời nhiều `k`.

In [12]:
def fib_bisect(k):
    cnt = 0
    while k:
        k -= FIBS[bisect_right(FIBS, k) - 1]
        cnt += 1
    return cnt

for k, want in examples:
    print(k, "->", fib_bisect(k), "| mong đợi", want)

7 -> 2 | mong đợi 2
10 -> 2 | mong đợi 2
19 -> 3 | mong đợi 3


In [13]:
class SolutionBisect:
    def findMinFibonacciNumbers(self, k: int) -> int:
        fibs = [1, 2]
        while fibs[-1] <= k:
            fibs.append(fibs[-1] + fibs[-2])
        cnt = 0
        while k:
            k -= fibs[bisect_right(fibs, k) - 1]
            cnt += 1
        return cnt

print([SolutionBisect().findMinFibonacciNumbers(k) for k, _ in examples])

[2, 2, 3]


### Cách 5 — Đi lùi trên dãy Fibonacci, bộ nhớ `O(1)`

Không cần lưu danh sách: đi lên bằng `(a, b) → (b, a + b)` tới khi `b > k` (lúc này `a` là Fibonacci lớn nhất ≤ k). Rồi **đi lùi** bằng `(a, b) → (b − a, a)` — phép ngược của bước đi lên. Ở mỗi bước lùi, nếu `a ≤ k` thì lấy `a`.

In [14]:
def fib_backward(k):
    a, b = 1, 1
    while b <= k:
        a, b = b, a + b          # sau vòng: a <= k < b
    cnt = 0
    while k:
        if a <= k:
            k -= a
            cnt += 1
        a, b = b - a, a          # lùi một bước trong dãy
    return cnt

for k, want in examples:
    print(k, "->", fib_backward(k), "| mong đợi", want)

7 -> 2 | mong đợi 2
10 -> 2 | mong đợi 2
19 -> 3 | mong đợi 3


Time `O(log k)`, Space `O(1)`. Lưu ý dãy lùi đi qua `…, 2, 1, 1, 0`; vòng lặp dừng khi `k == 0` — luôn xảy ra trước khi `a` về 0 vì số 1 có mặt.

## 8. TRACE từng bước

Trace Cách 5 trên `k = 19`:

| Giai đoạn | (a, b) | k | Lấy? | cnt |
|---|---|---|---|---|
| đi lên | (1,1)→(1,2)→(2,3)→(3,5)→(5,8)→(8,13)→(13,21) | 19 | | 0 |
| lùi | a = 13 | 19 → 6 | ✔ | 1 |
| lùi | a = 8 | 6 | ✘ | 1 |
| lùi | a = 5 | 6 → 1 | ✔ | 2 |
| lùi | a = 3 | 1 | ✘ | 2 |
| lùi | a = 2 | 1 | ✘ | 2 |
| lùi | a = 1 | 1 → 0 | ✔ | **3** |

In [15]:
def trace_backward(k):
    a, b = 1, 1
    ups = [(a, b)]
    while b <= k:
        a, b = b, a + b
        ups.append((a, b))
    print("đi lên:", " -> ".join(map(str, ups)))
    cnt = 0
    while k:
        if a <= k:
            print(f"  a={a:>3d}: k={k:>3d} -> {k - a:>3d} (lấy {a})")
            k -= a; cnt += 1
        else:
            print(f"  a={a:>3d}: k={k:>3d}  bỏ qua (a > k)")
        a, b = b - a, a
    print("đáp án:", cnt)
    return cnt

_ = trace_backward(19)

đi lên: (1, 1) -> (1, 2) -> (2, 3) -> (3, 5) -> (5, 8) -> (8, 13) -> (13, 21)
  a= 13: k= 19 ->   6 (lấy 13)
  a=  8: k=  6  bỏ qua (a > k)
  a=  5: k=  6 ->   1 (lấy 5)
  a=  3: k=  1  bỏ qua (a > k)
  a=  2: k=  1  bỏ qua (a > k)
  a=  1: k=  1 ->   0 (lấy 1)
đáp án: 3


Trace Cách 2 trên vài giá trị, in ra biểu diễn Zeckendorf và kiểm tra **không có hai số liền kề**:

In [16]:
def trace_greedy(k):
    fibs = fibs_upto(k)
    used = []
    rest = k
    for f in reversed(fibs):
        if f <= rest:
            used.append(f)
            rest -= f
            print(f"  lấy {f:>4d}, còn {rest}")
            if rest == 0:
                break
    idx = [fibs.index(u) for u in used]
    print(f"k={k}: {' + '.join(map(str, used))} | chỉ số trong dãy {idx}")
    return used

for k in [100, 1000]:
    trace_greedy(k)
    print()

  lấy   89, còn 11
  lấy    8, còn 3
  lấy    3, còn 0
k=100: 89 + 8 + 3 | chỉ số trong dãy [10, 5, 3]

  lấy  987, còn 13
  lấy   13, còn 0
k=1000: 987 + 13 | chỉ số trong dãy [15, 6]



## 9. Code nộp LeetCode

In [17]:
class Solution:
    def findMinFibonacciNumbers(self, k: int) -> int:
        a, b = 1, 1
        while b <= k:              # tìm Fibonacci lớn nhất <= k (là a)
            a, b = b, a + b
        count = 0
        while k:
            if a <= k:             # tham lam: lấy Fibonacci lớn nhất còn vừa
                k -= a
                count += 1
            a, b = b - a, a        # lùi xuống số Fibonacci liền trước
        return count

for k, want in examples:
    assert Solution().findMinFibonacciNumbers(k) == want
print("Solution đúng với 3 ví dụ của đề")

Solution đúng với 3 ví dụ của đề


## 10. Vì sao thuật toán đúng

Ký hiệu dãy Fibonacci "không trùng": `F₂ = 1, F₃ = 2, F₄ = 3, F₅ = 5, …`.

**Bổ đề 1 (chuẩn hoá).** Mọi biểu diễn `k` thành tổng Fibonacci có thể biến thành biểu diễn mà các số hạng **phân biệt** và **không liền kề**, với số số hạng **không tăng**.
- Nếu có hai số liền kề `Fₙ, Fₙ₊₁` ⇒ thay bằng `Fₙ₊₂` (giảm 1 số hạng).
- Nếu có số lặp `Fₙ, Fₙ` ⇒ thay bằng `Fₙ₊₁, Fₙ₋₂` (giữ nguyên số lượng; với `n = 2`: `1 + 1 = 2 = F₃`, giảm 1; với `n = 3`: `2 + 2 = 3 + 1`).
- Mỗi phép biến đổi làm "tổng chỉ số có trọng số" tăng hoặc số hạng giảm, nên quá trình dừng. Kết quả là biểu diễn chuẩn (Zeckendorf) với số hạng ≤ ban đầu.

**Bổ đề 2.** Tổng các Fibonacci phân biệt, không liền kề, có chỉ số lớn nhất `≤ m − 1` thì **< Fₘ**. (Quy nạp: tổng lớn nhất là `Fₘ₋₁ + Fₘ₋₃ + Fₘ₋₅ + … = Fₘ − 1`.)

**Định lý.** Gọi `Fₘ` = Fibonacci lớn nhất ≤ `k`. Lấy một lời giải tối ưu, chuẩn hoá theo Bổ đề 1 (vẫn tối ưu). Không số hạng nào > `k`, nên mọi số hạng ≤ `Fₘ`. Nếu `Fₘ` không có mặt, mọi số hạng có chỉ số ≤ `m − 1` ⇒ tổng < `Fₘ ≤ k` (Bổ đề 2) — mâu thuẫn. Vậy có lời giải tối ưu chứa `Fₘ` ⇒ `opt(k) = 1 + opt(k − Fₘ)`. Quy nạp theo `k` ⇒ tham lam cho đúng `opt(k)`.

**Vì sao mỗi số dùng ≤ 1 lần trong tham lam?** Vì `k < Fₘ₊₁` ⇒ `k − Fₘ < Fₘ₊₁ − Fₘ = Fₘ₋₁` ⇒ bước sau chọn số có chỉ số ≤ `m − 2`: không lặp, không liền kề — đúng là biểu diễn Zeckendorf.

**Cách 5:** phép lùi `(a, b) → (b − a, a)` đảo ngược đúng phép tiến `(a, b) → (b, a + b)`, nên `a` lần lượt đi qua mọi Fibonacci từ lớn đến nhỏ; tại mỗi số ta làm đúng quy tắc tham lam.

## 11. Phân tích Time / Space

Gọi `L` = số Fibonacci ≤ k ≈ `log_φ(k·√5)` ≈ 44 với `k = 10⁹` (`φ ≈ 1,618`).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. DP đổi tiền | `O(k · L)` | `O(k)` | chỉ `k` nhỏ (≤ ~10⁵) |
| 2. Tham lam lặp | `O(L)` | `O(L)` | |
| 3. Đệ quy + bisect | `O(L log L)` | `O(L)` | độ sâu ≤ L |
| 4. Bisect lặp | `O(L log L)` (danh sách dùng chung) | `O(L)` | nhiều truy vấn |
| 5. Đi lùi | `O(L)` | `O(1)` | gọn nhất |

Số số hạng của đáp án ≤ `⌈L/2⌉` (không liền kề) ⇒ tối đa ~22 với `k ≤ 10⁹`.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [18]:
ALL = {
    "dp": fib_dp,
    "greedy": fib_greedy,
    "rec": fib_rec,
    "bisect": fib_bisect,
    "bisect_submit": lambda k: SolutionBisect().findMinFibonacciNumbers(k),
    "backward": fib_backward,
    "submit": lambda k: Solution().findMinFibonacciNumbers(k),
}
edge = [
    (1, 1), (2, 1), (3, 1), (4, 2), (5, 1), (6, 2),
    (8, 1), (12, 3),                 # 12 = 8 + 3 + 1: số nhỏ nhất cần 3 số
    (13, 1), (33, 4),                # 33 = 21 + 8 + 3 + 1
    (100, 3), (1000, 2),             # 1000 = 987 + 13
]
for k, want in edge:
    assert fib_dp(k) == want, (k, fib_dp(k))
for name, f in ALL.items():
    for k, want in examples + edge:
        assert f(k) == want, (name, k, f(k))
print("Mọi cách đúng với", len(examples + edge), "test (nhãn edge kiểm chứng bằng DP)")

Mọi cách đúng với 15 test (nhãn edge kiểm chứng bằng DP)


### 12.2 So với DP đổi tiền trên **mọi** `k ≤ 3000` (một lần DP cho cả dải)

In [19]:
def dp_table(K):
    coins = sorted(set(fibs_upto(K)))
    dp = [0] + [K + 1] * K
    for x in range(1, K + 1):
        dp[x] = 1 + min(dp[x - c] for c in coins if c <= x)
    return dp

K = 3000
table = dp_table(K)
for k in range(1, K + 1):
    for name, f in ALL.items():
        if name == "dp":
            continue
        assert f(k) == table[k], (name, k)
print(f"mọi k từ 1 đến {K}: các cách tham lam khớp DP")

mọi k từ 1 đến 3000: các cách tham lam khớp DP


### 12.3 Ngẫu nhiên tới `10^9` (seed cố định): đối chiếu chéo + kiểm tra biểu diễn Zeckendorf hợp lệ

In [20]:
def is_zeckendorf(k, used):
    fibs = sorted(set(fibs_upto(max(k, 1))))
    idx = sorted(fibs.index(u) for u in used)
    distinct = len(set(used)) == len(used)
    non_adjacent = all(idx[i + 1] - idx[i] >= 2 for i in range(len(idx) - 1))
    return sum(used) == k and distinct and non_adjacent

rng = random.Random(1414)
t = time.perf_counter()
for _ in range(20000):
    k = rng.randint(1, 10**9)
    r = Solution().findMinFibonacciNumbers(k)
    assert r == fib_greedy(k) == fib_bisect(k) == fib_backward(k) == fib_rec(k)
    z = zeckendorf(k)
    assert len(z) == r and is_zeckendorf(k, z)
print(f"20000 giá trị ngẫu nhiên tới 1e9: các cách khớp, biểu diễn hợp lệ ({time.perf_counter() - t:.2f}s)")
print("k = 1e9 ->", Solution().findMinFibonacciNumbers(10**9), "=", " + ".join(map(str, zeckendorf(10**9))))

20000 giá trị ngẫu nhiên tới 1e9: các cách khớp, biểu diễn hợp lệ (0.48s)
k = 1e9 -> 14 = 701408733 + 267914296 + 24157817 + 5702887 + 514229 + 196418 + 75025 + 28657 + 1597 + 233 + 89 + 13 + 5 + 1


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Sinh Fibonacci **nhỏ hơn hẳn** `k` (bỏ sót trường hợp `k` là Fibonacci)

In [21]:
def wrong_strict_generation(k):
    fibs = []
    a, b = 1, 1
    while a < k:                                  # SAI: phải là a <= k
        fibs.append(a)
        a, b = b, a + b
    cnt = 0
    for f in reversed(fibs):
        while f <= k and k > 0:
            k -= f; cnt += 1
    return cnt

for k in [1, 8, 13, 7]:
    print(k, "| sai:", wrong_strict_generation(k), "| đúng:", Solution().findMinFibonacciNumbers(k))

1 | sai: 0 | đúng: 1
8 | sai: 2 | đúng: 1
13 | sai: 2 | đúng: 1
7 | sai: 2 | đúng: 2


`k = 8` là Fibonacci, đáp án 1, nhưng danh sách thiếu số 8 ⇒ `5 + 3` = 2. Với `k = 1` danh sách **rỗng** ⇒ trả 0. **Sửa:** điều kiện `a <= k`.

### Lỗi 2 — Lệch chỉ số khi dùng `bisect`

`bisect_left(fibs, k) − 1` cho phần tử **nhỏ hơn hẳn** `k`, không phải "lớn nhất ≤ k".

In [22]:
from bisect import bisect_left

def wrong_bisect_left(k):
    cnt = 0
    while k:
        i = bisect_left(FIBS, k) - 1              # SAI khi k là Fibonacci
        if i < 0:                                 # k = 1 -> i = -1 -> FIBS[-1] là số LỚN NHẤT!
            return f"hỏng: còn k=1 thì i=-1 (sau {cnt} bước)"
        k -= FIBS[i]
        cnt += 1
    return cnt

for k in [7, 8, 13, 21, 22]:
    print(k, "| sai:", wrong_bisect_left(k), "| đúng:", Solution().findMinFibonacciNumbers(k))

7 | sai: hỏng: còn k=1 thì i=-1 (sau 2 bước) | đúng: 2
8 | sai: hỏng: còn k=1 thì i=-1 (sau 2 bước) | đúng: 1
13 | sai: hỏng: còn k=1 thì i=-1 (sau 3 bước) | đúng: 1
21 | sai: hỏng: còn k=1 thì i=-1 (sau 3 bước) | đúng: 1
22 | sai: hỏng: còn k=1 thì i=-1 (sau 1 bước) | đúng: 2


Với `k = 13`: `bisect_left` trả vị trí của 13, trừ 1 ⇒ lấy **8** thay vì 13 (đúng ra chỉ cần 1 số); cứ mỗi lần phần còn lại là một số Fibonacci, bản sai lại chọn số nhỏ hơn. Tệ hơn, khi phần còn lại là **1** (hay gặp!), `bisect_left` = 0, trừ 1 ⇒ chỉ số `−1` ⇒ `FIBS[−1]` là số **lớn nhất** trong danh sách ⇒ `k` âm và vòng lặp không bao giờ dừng. Bản minh hoạ chặn trường hợp này và báo "hỏng" thay vì treo. **Sửa:** `bisect_right(FIBS, k) − 1`.

### Lỗi 3 — Tin rằng tham lam đúng với **mọi** hệ mệnh giá

Sau bài này dễ "quen tay" dùng tham lam cho bài đổi tiền khác. Phản ví dụ với mệnh giá `{1, 3, 4}`:

In [23]:
def greedy_coins(coins, k):
    cnt = 0
    for c in sorted(coins, reverse=True):
        cnt += k // c
        k %= c
    return cnt

for coins, k in [([1, 3, 4], 6), ([1, 5, 6, 9], 11), ([1, 2, 3, 5, 8, 13], 19)]:
    print(coins, k, "| tham lam:", greedy_coins(coins, k), "| DP tối ưu:", min_coins(coins, k))

[1, 3, 4] 6 | tham lam: 3 | DP tối ưu: 2
[1, 5, 6, 9] 11 | tham lam: 3 | DP tối ưu: 2
[1, 2, 3, 5, 8, 13] 19 | tham lam: 3 | DP tối ưu: 3


Với `{1, 3, 4}`, `k = 6`: tham lam `4 + 1 + 1` (3 đồng), tối ưu `3 + 3` (2 đồng). Với Fibonacci thì hai cột trùng nhau. **Bài học:** tham lam chỉ dùng khi **chứng minh được** (ở đây nhờ Zeckendorf) hoặc đã kiểm chứng bằng DP.

### Lỗi 4 — Sinh số Fibonacci bằng đệ quy không nhớ

`fib(n) = fib(n−1) + fib(n−2)` gọi lại hàm theo cấp số nhân. Đếm số lời gọi (không chạy tới `n` lớn để notebook không treo):

In [24]:
calls = 0
def slow_fib(n):
    global calls
    calls += 1
    return 1 if n <= 2 else slow_fib(n - 1) + slow_fib(n - 2)

for n in [10, 20, 25]:
    calls = 0
    v = slow_fib(n)
    print(f"slow_fib({n}) = {v}, số lời gọi = {calls}")
print("cần tới F_44 ≈ 7·10^8 -> khoảng", f"{2 * 701408733 - 1:,}", "lời gọi: không khả thi")

slow_fib(10) = 55, số lời gọi = 109
slow_fib(20) = 6765, số lời gọi = 13529
slow_fib(25) = 75025, số lời gọi = 150049
cần tới F_44 ≈ 7·10^8 -> khoảng 1,402,817,465 lời gọi: không khả thi


**Sửa:** sinh lặp với hai biến (mục 3.1), hoặc `lru_cache`.

### Lỗi 5 — Đi lùi sai công thức trong Cách 5

Viết `a, b = b, b − a` (đảo nhầm vị trí) thì dãy lùi không còn là Fibonacci.

In [25]:
def wrong_backward(k, limit=100):
    a, b = 1, 1
    while b <= k:
        a, b = b, a + b
    cnt, steps = 0, 0
    while k and steps < limit:
        steps += 1
        if 0 < a <= k:
            k -= a; cnt += 1
        a, b = b, b - a                            # SAI: phải là (b - a, a)
    return cnt if k == 0 else f"kẹt (k còn {k})"

for k in [7, 10, 19, 100]:
    print(k, "| sai:", wrong_backward(k), "| đúng:", Solution().findMinFibonacciNumbers(k))

7 | sai: kẹt (k còn 2) | đúng: 2
10 | sai: kẹt (k còn 2) | đúng: 2
19 | sai: kẹt (k còn 6) | đúng: 3
100 | sai: kẹt (k còn 11) | đúng: 3


**Sửa:** từ `(a, b)` với `b = a + a_trước` ⇒ số trước là `b − a` ⇒ cặp mới `(b − a, a)`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Ít số hạng nhất có tổng bằng k"** ⇒ coin change; nếu `k` lớn (10⁹) ⇒ chắc chắn có tham lam/công thức.
- Mệnh giá Fibonacci, luỹ thừa của 2, hệ tiền "chuẩn" (1, 2, 5, 10, …) ⇒ tham lam đúng; hệ tuỳ ý ⇒ DP.
- Dãy tăng theo cấp số nhân ⇒ chỉ có `O(log k)` phần tử ⇒ có thể sinh hết rồi duyệt.
- Kiểm chứng tham lam: viết DP cho `k` nhỏ, so **toàn bộ** dải.
- Muốn `O(1)` bộ nhớ trên dãy truy hồi ⇒ đi tiến rồi **đi lùi** bằng phép ngược.

## 15. Biến thể của bài

1. **In ra các số được dùng** (biểu diễn Zeckendorf) — Bài 4 mục 17.
2. **Số `k` nhỏ nhất cần đúng `t` số Fibonacci** ⇒ `F₂ₜ₊₁ − 1` (1, 4, 12, 33, 88, …) — Bài 5.
3. **Mã Fibonacci (Fibonacci coding)**: mã hoá số nguyên bằng chuỗi bit Zeckendorf + bit `1` kết thúc.
4. **Đếm số cách** viết `k` thành tổng Fibonacci (không thứ tự) ⇒ DP đếm kiểu coin change.
5. **Mỗi số Fibonacci dùng tối đa một lần** ⇒ đáp án không đổi (biểu diễn Zeckendorf đã phân biệt).
6. Bài họ hàng: LeetCode 322 *Coin Change* (DP), 991 *Broken Calculator* (tham lam ngược), 1342 *Number of Steps to Reduce a Number to Zero*.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. DP đổi tiền | `dp[x] = 1 + min dp[x − F]` | `O(k·L)` | `O(k)` | kiểm chứng với `k` nhỏ; hệ tiền bất kỳ |
| 2. Tham lam lặp | duyệt Fibonacci từ lớn đến nhỏ | `O(L)` | `O(L)` | **mặc định**, dễ hiểu |
| 3. Đệ quy | `1 + f(k − F_max)` | `O(L log L)` | `O(L)` | đúng như gợi ý đề |
| 4. Bisect lặp | danh sách dùng chung, `bisect_right` | `O(L log L)` | `O(L)` | nhiều truy vấn `k` |
| 5. Đi lùi | `(a, b) → (b − a, a)` | `O(L)` | `O(1)` | tối ưu bộ nhớ, gọn |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
`findMinFibonacciNumbers(100)` và `findMinFibonacciNumbers(987)` bằng bao nhiêu?

### Bài 2 (tính tay)
Tìm biểu diễn Zeckendorf của `50` và của `143`; cho biết đáp án của mỗi số.

### Bài 3 (điền TODO)
Hoàn thiện `greedy_todo(k)` (Cách 2).

### Bài 4 (tự viết lại)
Viết `fib_terms(k)` trả về **danh sách** các số Fibonacci (giảm dần) trong một lời giải tối ưu, và kiểm tra chúng phân biệt, không liền kề.

### Bài 5 (mini challenge)
Viết `smallest_needing(t)`: số `k` **nhỏ nhất** cần đúng `t` số Fibonacci. Đoán công thức, rồi kiểm chứng bằng cách quét `k` tăng dần với `Solution`.

In [26]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def greedy_todo(k):
    fibs = fibs_upto(k)
    cnt = 0
    for f in reversed(fibs):
        # TODO: nếu f <= k thì trừ và đếm
        # TODO: dừng khi k == 0
        pass
    return cnt

# Bài 4 — TODO
def fib_terms(k):
    # TODO: như greedy nhưng lưu các số đã lấy
    pass

# Bài 5 — TODO
def smallest_needing(t):
    # TODO: thử công thức F_{2t+1} - 1 hoặc cộng dồn F_2 + F_4 + ... + F_{2t}
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Fibonacci lớn nhất ≤ 100 là 89, còn 11. (2) 11 → 8, còn 3 → 3. (3) Đáp án 3; `987` là Fibonacci ⇒ 1.

**Bài 2.** (1) `50 − 34 = 16`, `16 − 13 = 3`, `3` ⇒ `34 + 13 + 3` (3 số). (2) `143 − 89 = 54`, `54 − 34 = 20`, `20 − 13 = 7`, `7 − 5 = 2`. (3) `143 = 89 + 34 + 13 + 5 + 2` ⇒ 5 số.

**Bài 3.** (1) `if f <= k: k -= f; cnt += 1`. (2) `if k == 0: break`. (3) So với `Solution` trên `k` ngẫu nhiên.

**Bài 4.** (1) `used.append(f)` thay cho `cnt += 1`. (2) Kiểm tra `sum(used) == k`. (3) Vị trí trong dãy `F₂, F₃, …` cách nhau ≥ 2.

**Bài 5.** (1) Quét: `t = 1` ⇒ 1, `t = 2` ⇒ 4, `t = 3` ⇒ 12, `t = 4` ⇒ 33. (2) Nhận ra `1 = 1`, `4 = 1 + 3`, `12 = 1 + 3 + 8`, `33 = 1 + 3 + 8 + 21` — cộng dồn Fibonacci "cách một". (3) Tổng đó bằng `F₂ₜ₊₁ − 1` (2 − 1, 5 − 1, 13 − 1, 34 − 1).

## 19. Đáp án tham khảo

In [27]:
# Bài 1, 2
assert Solution().findMinFibonacciNumbers(100) == 3
assert Solution().findMinFibonacciNumbers(987) == 1
assert zeckendorf(50) == [34, 13, 3] and Solution().findMinFibonacciNumbers(50) == 3
assert zeckendorf(143) == [89, 34, 13, 5, 2] and Solution().findMinFibonacciNumbers(143) == 5
print("Bài 1, 2: đúng")

Bài 1, 2: đúng


In [28]:
# Bài 3 — đáp án
def greedy_todo_sol(k):
    fibs = fibs_upto(k)
    cnt = 0
    for f in reversed(fibs):
        if f <= k:
            k -= f
            cnt += 1
            if k == 0:
                break
    return cnt

rng = random.Random(3)
for _ in range(2000):
    k = rng.randint(1, 10**9)
    assert greedy_todo_sol(k) == Solution().findMinFibonacciNumbers(k)
for k in range(1, 3001):
    assert greedy_todo_sol(k) == table[k]
print("Bài 3: đúng")

Bài 3: đúng


In [29]:
# Bài 4 — đáp án
def fib_terms_sol(k):
    a, b = 1, 1
    while b <= k:
        a, b = b, a + b
    used = []
    while k:
        if a <= k:
            used.append(a)
            k -= a
        a, b = b - a, a
    return used

print(fib_terms_sol(19), fib_terms_sol(10**9))
rng = random.Random(4)
for _ in range(2000):
    k = rng.randint(1, 10**9)
    u = fib_terms_sol(k)
    assert is_zeckendorf(k, u) and len(u) == Solution().findMinFibonacciNumbers(k)
print("Bài 4: đúng")

[13, 5, 1] [701408733, 267914296, 24157817, 5702887, 514229, 196418, 75025, 28657, 1597, 233, 89, 13, 5, 1]
Bài 4: đúng


In [30]:
# Bài 5 — đáp án
def smallest_needing_sol(t):
    a, b = 1, 1                  # F1, F2
    for _ in range(2 * t - 1):   # tiến tới cặp (F_{2t}, F_{2t+1})
        a, b = b, a + b
    return b - 1                 # F_{2t+1} - 1

first = {}
k = 1
while len(first) < 6:
    r = Solution().findMinFibonacciNumbers(k)
    first.setdefault(r, k)
    k += 1
print("quét:", dict(sorted(first.items())))
for t in range(1, 7):
    assert smallest_needing_sol(t) == first[t]
print("công thức cho t = 1..10:", [smallest_needing_sol(t) for t in range(1, 11)])
for t in range(1, 12):
    s = smallest_needing_sol(t)
    assert Solution().findMinFibonacciNumbers(s) == t and Solution().findMinFibonacciNumbers(s - 1) < t
print("Bài 5: đúng")

quét: {1: 1, 2: 4, 3: 12, 4: 33, 5: 88, 6: 232}
công thức cho t = 1..10: [1, 4, 12, 33, 88, 232, 609, 1596, 4180, 10945]
Bài 5: đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template tham lam Fibonacci (bộ nhớ O(1)):**

```python
a, b = 1, 1
while b <= k:          # a = Fibonacci lớn nhất <= k
    a, b = b, a + b
cnt = 0
while k:
    if a <= k:
        k -= a; cnt += 1
    a, b = b - a, a    # lùi một bước
return cnt
```

**Template với danh sách:** `fibs` sinh với điều kiện `<= k`; lớn nhất ≤ x là `fibs[bisect_right(fibs, x) - 1]`.

**Checklist trước khi submit:**
- [ ] Sinh Fibonacci với điều kiện `<= k` (để `k` là Fibonacci cho đáp án 1).
- [ ] Dùng `bisect_right(...) - 1`, không phải `bisect_left(...) - 1`.
- [ ] Không dùng DP mảng kích thước `k` (k tới 10⁹).
- [ ] Đi lùi đúng công thức `(b − a, a)`.
- [ ] Đã thử `k = 1, 2, 4, 12, 19, 10⁹`.
- [ ] Đúng chữ ký: `findMinFibonacciNumbers(self, k: int) -> int`.